# Embedding Generation

Embeds `tovima_nlp_ready.csv` (column `embedding_text`) with BGE-M3 via
Ollama on CPU. Output: `tovima_embeddings_bge_m3.npy`.

Modelfile:

    FROM bge-m3
    PARAMETER num_ctx 8192
    PARAMETER num_batch 8192
    PARAMETER num_gpu 0
    TEMPLATE {{ .Prompt }}

In [6]:
import numpy as np
import pandas as pd
import ollama
import json
from tqdm import tqdm

## 1. Load

In [7]:
df = pd.read_csv("tovima_nlp_ready.csv", encoding="utf-32", sep="\t")
df["to_lists"] = df["to_lists"].apply(json.loads)
df["to_other_recipients"] = df["to_other_recipients"].apply(json.loads)

print("Rows:", len(df))
print("Columns:", df.columns.tolist())

Rows: 13637
Columns: ['Author', 'Date', 'To', 'Subject', 'Message', 'Date_parsed', 'Year', 'had_html_markup', 'is_reply_or_forward', 'to_lists', 'to_other_recipients', 'subject_tag', 'contains_pii_pattern', 'embedding_text', 'cleaned_text']


## 2. Verify model is loaded and on CPU

In [8]:
status = ollama.ps()
print(status)

# expect 100% CPU, size_vram = 0


models=[]


## 3. Embed

In [9]:
MAX_CHARS = 7000
MODEL = "bge-m3-dimiper"

texts = df["embedding_text"].fillna("").tolist()

n_truncated = sum(1 for t in texts if len(t) > MAX_CHARS)
print(f"Texts to embed: {len(texts)}")
print(f"Will be truncated at {MAX_CHARS} chars: {n_truncated} ({n_truncated/len(texts)*100:.1f}%)")

Texts to embed: 13637
Will be truncated at 7000 chars: 511 (3.7%)


In [10]:
embeddings = []
errors = []

for i, text in enumerate(tqdm(texts, desc="Embedding")):
    try:
        truncated = text[:MAX_CHARS]
        response = ollama.embeddings(model=MODEL, prompt=truncated)
        embeddings.append(response["embedding"])
    except Exception as e:
        # record failures
        print(f"\nError at row {i}: {e}")
        errors.append(i)
        embeddings.append(None)

print(f"\nDone. Errors: {len(errors)}")
if errors:
    print("Error indices:", errors)

Embedding: 100%|██████████| 13637/13637 [2:10:57<00:00,  1.74it/s]  


Done. Errors: 0


## 4. Handle any errors

In [11]:
# inspect failed rows
if errors:
    for idx in errors:
        print(f"row {idx}: {repr(texts[idx][:100])}")
else:
    print("All rows embedded successfully.")

All rows embedded successfully.


## 5. Validate and save

In [12]:
# drop failed rows
valid_mask = [e is not None for e in embeddings]
n_valid = sum(valid_mask)
n_dropped = len(valid_mask) - n_valid

print(f"Valid embeddings: {n_valid} / {len(embeddings)}")
if n_dropped > 0:
    print(f"Dropped {n_dropped} rows that failed to embed")

df_final = df[valid_mask].reset_index(drop=True)
embeddings_clean = [e for e in embeddings if e is not None]

embeddings_array = np.array(embeddings_clean, dtype=np.float32)
print(f"Embedding array shape: {embeddings_array.shape}")
print(f"Vector dimension: {embeddings_array.shape[1]}")

# sanity checks
assert not np.isnan(embeddings_array).any(), "NaN values found in embeddings"
assert not (embeddings_array == 0).all(axis=1).any(), "All-zero vectors found"
print("Sanity checks passed.")

Valid embeddings: 13637 / 13637
Embedding array shape: (13637, 1024)
Vector dimension: 1024
Sanity checks passed.


In [13]:
# save
df_out = df_final.copy()
df_out["to_lists"] = df_out["to_lists"].apply(json.dumps)
df_out["to_other_recipients"] = df_out["to_other_recipients"].apply(json.dumps)

df_out.to_csv("tovima_embedded.csv", sep="\t", encoding="utf-32", index=False)
np.save("tovima_embeddings_bge_m3.npy", embeddings_array)

print(f"Saved tovima_embedded.csv: {df_out.shape}")
print(f"Saved tovima_embeddings_bge_m3.npy: {embeddings_array.shape}")


Saved tovima_embedded.csv: (13637, 15)
Saved tovima_embeddings_bge_m3.npy: (13637, 1024)
